In [2]:

import requests
from bs4 import BeautifulSoup
from collections import defaultdict
# Target URL
URL = "http://abulafia.mt.ic.ac.uk/shannon/radius.php"

# Send request
response = requests.get(URL)
response.raise_for_status()

# Parse HTML
soup = BeautifulSoup(response.text, "lxml")

# Find the table (main ionic radii table)
table = soup.find("table")


# element → charge → coordination → radius
elements = dict()

rows = table.find_all("tr") # skip header

for row in rows:

    cols = [c.get_text(strip=True) for c in row.find_all("td")]
    
    if cols==['Ion', 'Charge', 'Coordination', 'Spin State', 'Crystal Radius', 'Ionic Radius', 'Key*']:
        continue

    # Ion | Charge | Coordination | Spin | Crystal R | Ionic R | Key
    if len(cols) > 6:
        element = cols[0]
        charge = cols[1]
        coordination = cols[2]
        ionic_radius = float(cols[5])

        elements[element]={"Charge":{charge:{coordination:ionic_radius}}}

    elif len(cols)>5:
        charge = cols[0]
        coordination = cols[1]
        ionic_radius = float(cols[4])
        elements[element]["Charge"][charge]={coordination:ionic_radius}
    elif len(cols)==5:
        coordination = cols[0]
        ionic_radius = float(cols[3])
        elements[element]["Charge"][charge][coordination]=ionic_radius
    else:
        continue


#fixing C_SITE values
C_SITE=set(["O","F","Br","I","Cl"])
for i in C_SITE:
    if i=="O":
        elements[i]={"Charge":{"-2":{"VI":elements[i]["Charge"]["-2"]["VI"]}}}
    else:
        elements[i]={"Charge":{"-1":{"VI":elements[i]["Charge"]["-1"]["VI"]}}}


In [3]:
print(elements["Ag"])

{'Charge': {'1': {'II': 0.67, 'IV': 1.0, 'IVSQ': 1.02, 'V': 1.09, 'VI': 1.15, 'VII': 1.22, 'VIII': 1.28}, '2': {'IVSQ': 0.79, 'VI': 0.94}, '3': {'IVSQ': 0.67, 'VI': 0.75}}}


PREPROCESSING OF ALL POSSIBLE COMPOUNDS THAT FOLLOW THE PEROVSKITE FORMULAS

In [4]:
import pandas as pd
import regex as re

data=pd.read_csv("INPUT DATA/mp_perovskites2.csv")

In [5]:
#identifying perovskite type
def identify_ions(row):
    formula=row["formula_pretty"]
    # Regex: Matches Capital Letter (+ lowercase) followed by optional digits/decimals
    tokens = re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', formula)
    # Process tokens: If count is empty, it's 1.0 (e.g., 'Cs' -> 1)
    ions = []
    counts = []
    for el, count in tokens:
        ions.append(el)
        counts.append(float(count) if count else 1.0)
    
    num_elements = len(ions)
    # Pattern Logic
    # 1. ABC3 or A2BC6: 3 elements, counts are [1, 1, 3] or [2,1,6] in any order
    if num_elements == 3:
        if counts== [1.0, 1.0, 3.0] or counts==[2.0,1.0,6.0]:
            return [ions[0],ions[1],ions[2]]
            
    # 2. A2BCD6: 4 elements, counts are [1, 1, 2, 6] in any order
    elif num_elements == 4:
        if counts == [2.0, 1.0, 1.0, 6.0]:
            return [ions[0],ions[1]+"; "+ions[2],ions[3]]
            
    # 3. ABCDE6 or A2BCD3E3: 5 elements, counts are [1, 1, 1, 1, 6] or [2, 1, 1, 3, 3] in any order
    elif num_elements == 5:
        if counts == [1.0, 1.0, 1.0, 1.0, 6.0]:
            return [ions[0]+"; "+ions[1],ions[2]+"; "+ions[3],ions[4]]
        if counts == [2.0, 1.0, 1.0, 3.0, 3.0]:
            return [ions[0], ions[1]+"; "+ions[2],ions[3]+"; "+ions[4]]
    
    return None



In [6]:
from itertools import product

def perovskite_radii(row):
    ions=row["ions"]
    A=ions[0].split("; ")
    B=ions[1].split("; ")
    C=ions[2].split("; ")
    
    a_sites=[]
    b_sites=[]
    c_sites=[]
    formula=dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["formula_pretty"]))
    if len(A+B+C)==3:
        if formula[A[0]]==1:   #ABC3
            a_sites=[(A[0],1)]
            b_sites=[(B[0],1)]
            c_sites=[(C[0],3)]
        else:               #A2BC6
            a_sites=[(A[0],2)]
            b_sites=[(B[0],1)]
            c_sites=[(C[0],6)]
    
    elif len(A+B+C)==4:     #A2BB'C6
            a_sites=[(A[0],2)]
            b_sites=[(B[0],1),(B[1],1)]
            c_sites=[(C[0],6)]

    else:
        if len(A)==2:       #AA'BB'C6
            a_sites=[(A[0],1),(A[1],1)]
            b_sites=[(B[0],1),(B[1],1)]
            c_sites=[(C[0],6)]
        else:               #A2BB'C3C'3
            a_sites=[(A[0],2)]
            b_sites=[(B[0],1),(B[1],1)]
            c_sites=[(C[0],3),(C[1],3)]
    """
    Calculates ionic radii for A, B, and C sites ensuring charge neutrality 
    based on the stoichiometric coefficients of each element.
    
    Parameters:
    - a_sites: List of tuples (ion_name, count) e.g., [("Sr", 1), ("La", 1)]
    - b_sites: List of tuples (ion_name, count) e.g., [("Fe", 1), ("Sn", 1)]
    - c_sites: List of tuples (ion_name, count) e.g., [("O", 6)]
    - elements: The dictionary containing Shannon radii data.
    """
    
    # Helper to validate elements and get available charges
    def get_valid_charges(site_list):
        site_charges = []
        for ion, count in site_list:
            if ion not in elements:
                return None
            site_charges.append(list(elements[ion]['Charge'].keys()))
        return site_charges

    a_charge_options = get_valid_charges(a_sites)
    b_charge_options = get_valid_charges(b_sites)
    c_charge_options = get_valid_charges(c_sites)

    if None in [a_charge_options, b_charge_options, c_charge_options]:
        return None

    # Iterate through all combinations of oxidation states
    for a_comb in product(*a_charge_options):
        for b_comb in product(*b_charge_options):
            for c_comb in product(*c_charge_options):
                
                # Calculate stoichiometric charge sum: Σ (charge * coefficient)
                q_a = sum(float(charge) * count for charge, (_, count) in zip(a_comb, a_sites))
                q_b = sum(float(charge) * count for charge, (_, count) in zip(b_comb, b_sites))
                q_c = sum(float(charge) * count for charge, (_, count) in zip(c_comb, c_sites))
                
                if abs(q_a + q_b + q_c)==0:
                    try:
                        # Coordination requirements: A=XII, B=VI, C=VI
                        # Organic compounds (added to elements_dict) will have 'XII'
                        radii_a = [elements[ion]['Charge'][char].get('XII') 
                                   for (ion, _), char in zip(a_sites, a_comb)]
                        
                        radii_b = [elements[ion]['Charge'][char].get('VI') 
                                   for (ion, _), char in zip(b_sites, b_comb)]
                        
                        radii_c = [elements[ion]['Charge'][char].get('VI') 
                                   for (ion, _), char in zip(c_sites, c_comb)]
                        
                        # Verify that radii exist for the required coordination
                        if None not in radii_a + radii_b + radii_c:
                            return [radii_a,radii_b,radii_c]
                    except KeyError:
                        continue
                        
    return None


In [7]:
import math
def tolerance_factor(row):
    rA=sum(row["A_radii"])/len(row["A_radii"])
    rB=sum(row["B_radii"])/len(row["B_radii"])
    rC=sum(row["C_radii"])/len(row["C_radii"])
    return (rA + rC) / (math.sqrt(2) * (rB + rC))

def octahedral_factor(row):
    rA=sum(row["A_radii"])/len(row["A_radii"])
    rB=sum(row["B_radii"])/len(row["B_radii"])
    rC=sum(row["C_radii"])/len(row["C_radii"])
    return rB / rC

In [8]:
data["ions"]=data.apply(identify_ions,axis=1)
# Show rows where ion identification failed
#print(data[data["ions"].isnull()]["formula_pretty"])

# Drop them permanently before filtering by C_SITE
data = data.dropna(subset=["ions"])
print(data.shape)
#allowed C-site elements
C_SITE = ["O", "F", "Br", "I", "Cl"]


#lambda to check index 2 (the C-site) of each 'ions' tuple
data = data[data["ions"].apply(
    lambda x: all(ion.strip() in C_SITE for ion in x[2].split(";"))
)]
print(data.shape)
data["radii"]=data.apply(perovskite_radii,axis=1)
data = data.dropna(subset=['ions',"radii"])

data[['A_ions', 'B_ions', 'C_ions']] = pd.DataFrame(data['ions'].tolist(), index=data.index)
data[['A_radii', 'B_radii', 'C_radii']] = pd.DataFrame(data['radii'].tolist(), index=data.index)

data["Tolerance_factor"]=data.apply(tolerance_factor,axis=1)
data["Octahedral_factor"]=data.apply(octahedral_factor,axis=1)

data.drop(columns=["radii","ions",'A_radii', 'B_radii', 'C_radii'],inplace=True)

(10587, 9)
(9722, 9)


In [9]:
print(data.shape)
data.to_csv("INPUT DATA/perovskite2.csv")

(4585, 13)


Series([], Name: formula_pretty, dtype: object)
(10587, 9)


In [14]:
import pandas as pd
import numpy as np
import re
import math
from itertools import product

# ==========================================
# 1. CORE HELPERS
# ==========================================

def can_have_cn(element, cn_target, elements_dict):
    """Checks if an element supports a specific coordination number (e.g., XII or VI)."""
    if element not in elements_dict: return False
    charges = elements_dict[element].get('Charge', {})
    for charge_val in charges.values():
        if cn_target in charge_val: return True
    return False

def identify_ions(row, elements_dict):
    """Parses formula and classifies elements into A, B, and C sites."""
    formula = str(row["formula_pretty"]).replace(" ", "")
    tokens = re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', formula)
    data = [(el, float(count) if count else 1.0) for el, count in tokens]
    
    # 1. Identify Anions (C-sites)
    known_anions = {'O', 'F', 'Cl', 'Br', 'I', 'S', 'Se', 'Te'}
    c_ions = [el for el, c in data if c >= 3.0 or el in known_anions]
    
    # 2. Identify Cations and classify based on CN capabilities
    cations = [(el, c) for el, c in data if el not in c_ions]
    a_sites, b_sites = [], []
    
    for el, count in cations:
        is_a = can_have_cn(el, 'XII', elements_dict)
        is_b = can_have_cn(el, 'VI', elements_dict)
        
        if is_a and not is_b:
            a_sites.append(el)
        elif is_b and not is_a:
            b_sites.append(el)
        else:
            # Tie-break based on stoichiometry or priority
            if count == 2.0 or not a_sites:
                a_sites.append(el)
            else:
                b_sites.append(el)
                    
    return pd.Series({
        'A_ions': "; ".join(a_sites) if a_sites else "Unknown",
        'B_ions': "; ".join(b_sites) if b_sites else "Unknown",
        'C_ions': "; ".join(c_ions) if c_ions else "Unknown"
    })

# ==========================================
# 2. RADII & STRUCTURAL FACTORS
# ==========================================

def perovskite_radii(row, elements_dict):
    """Calculates stoichiometric ionic radii ensuring charge neutrality."""
    # Split semicolon-separated strings from the dataframe columns
    A_list = str(row['A_ions']).split("; ")
    B_list = str(row['B_ions']).split("; ")
    C_list = str(row['C_ions']).split("; ")
    
    # Regex to capture element and stoichiometry
    formula_map = dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["formula_pretty"]))
    
    # Helper to safely convert '' or missing values to 1.0
    def get_stoich(el):
        val = formula_map.get(el, 1.0)
        if val == '' or val is None:
            return 1.0
        try:
            return float(val)
        except ValueError:
            return 1.0

    # Helper for safe indexing to prevent IndexError
    def get_el(lst, idx): 
        return lst[idx] if len(lst) > idx else lst[0]

    # Determine structure type and assign site counts
    total_elements = len(A_list + B_list + C_list)
    
    if total_elements == 3:
        if get_stoich(A_list[0]) == 1.0: # ABC3
            a_sites, b_sites, c_sites = [(A_list[0], 1.0)], [(B_list[0], 1.0)], [(C_list[0], 3.0)]
        else: # A2BC6
            a_sites, b_sites, c_sites = [(A_list[0], 2.0)], [(B_list[0], 1.0)], [(C_list[0], 6.0)]
            
    elif total_elements == 4: # A2BB'C6
        a_sites = [(A_list[0], 2.0)]
        b_sites = [(B_list[0], 1.0), (get_el(B_list, 1), 1.0)]
        c_sites = [(C_list[0], 6.0)]
        
    else: # AA'BB'C6 or A2BB'C3C'3
        if len(A_list) >= 2: # AA'BB'C6
            a_sites = [(A_list[0], 1.0), (A_list[1], 1.0)]
            b_sites = [(B_list[0], 1.0), (get_el(B_list, 1), 1.0)]
            c_sites = [(C_list[0], 6.0)]
        else: # A2BB'C3C'3
            a_sites = [(A_list[0], 2.0)]
            b_sites = [(B_list[0], 1.0), (get_el(B_list, 1), 1.0)]
            c_sites = [(C_list[0], 3.0), (get_el(C_list, 1), 3.0)]

    def get_valid_charges(site_list):
        opts = []
        for ion, _ in site_list:
            if ion not in elements_dict: return None
            opts.append(list(elements_dict[ion]['Charge'].keys()))
        return opts

    a_opts = get_valid_charges(a_sites)
    b_opts = get_valid_charges(b_sites)
    c_opts = get_valid_charges(c_sites)

    if None in [a_opts, b_opts, c_opts]: return None

    # Search for Charge Neutrality (q_total == 0)
    for a_comb, b_comb, c_comb in product(product(*a_opts), product(*b_opts), product(*c_opts)):
        q_a = sum(float(q) * c for q, (_, c) in zip(a_comb, a_sites))
        q_b = sum(float(q) * c for q, (_, c) in zip(b_comb, b_sites))
        q_c = sum(float(q) * c for q, (_, c) in zip(c_comb, c_sites))
        
        if math.isclose(q_a + q_b + q_c, 0, abs_tol=1e-5):
            try:
                rad_a = [elements_dict[ion]['Charge'][q].get('XII') for (ion, _), q in zip(a_sites, a_comb)]
                rad_b = [elements_dict[ion]['Charge'][q].get('VI') for (ion, _), q in zip(b_sites, b_comb)]
                rad_c = [elements_dict[ion]['Charge'][q].get('VI') for (ion, _), q in zip(c_sites, c_comb)]
                
                if None not in (rad_a + rad_b + rad_c):
                    return [rad_a, rad_b, rad_c]
            except KeyError: continue
    return None


def tolerance_factor(row):
    rA, rB, rC = np.mean(row["A_radii"]), np.mean(row["B_radii"]), np.mean(row["C_radii"])
    return (rA + rC) / (math.sqrt(2) * (rB + rC))

def octahedral_factor(row):
    rB, rC = np.mean(row["B_radii"]), np.mean(row["C_radii"])
    return rB / rC

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================

# 1. Load and Identify
data = pd.read_csv("INPUT DATA/mp_perovskites2.csv")
data[['A_ions', 'B_ions', 'C_ions']] = data.apply(identify_ions, axis=1, args=(elements,), result_type='expand')

# 2. Filter Anions
C_SITE_ALLOWED = ["O", "F", "Br", "I", "Cl"]
data = data[data["C_ions"].apply(lambda x: all(ion.strip() in C_SITE_ALLOWED for ion in str(x).split(";")))]

# 3. Radii Calculation
# We pass the 'elements' dictionary as an argument here
data["radii_tuple"] = data.apply(perovskite_radii, axis=1, args=(elements,))
data = data.dropna(subset=["radii_tuple"])

# 4. Unpack and Structural Factors
data[['A_radii', 'B_radii', 'C_radii']] = pd.DataFrame(data['radii_tuple'].tolist(), index=data.index)
data["Tolerance_factor"] = data.apply(tolerance_factor, axis=1)
data["Octahedral_factor"] = data.apply(octahedral_factor, axis=1)

# 5. Volume Normalization
for i, row in data.iterrows():
    formula = dict(re.findall(r'([A-Z][a-z]*)(\d*\.?\d*)', row["formula_pretty"]))
    total_stoich = sum(float(c) if c else 1.0 for c in formula.values())
    ratio = int(row["nsites"]) / total_stoich
    data.at[i, "volume"] = row["volume"] / ratio

# 6. Final Cleanup
data.drop(columns=["radii_tuple", "A_ions", "B_ions", "C_ions", "A_radii", "B_radii", "C_radii"], inplace=True)
print(f"Success! Final shape: {data.shape}")

Success! Final shape: (5045, 10)
